In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             classification_report, confusion_matrix, RocCurveDisplay)
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ── 1. Download Data ──────────────────────────────────────────────────────────
print("Downloading S&P 500 data...")
df = yf.download("^GSPC", start="2010-01-01", end="2023-12-31", auto_adjust=True)

# Flatten MultiIndex columns returned by newer yfinance versions
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df = df[["Open", "High", "Low", "Close", "Volume"]].dropna()

# Ensure all columns are 1D Series
df = df.apply(lambda col: col.squeeze())

print(f"  {len(df)} trading days loaded ({df.index[0].date()} to {df.index[-1].date()})")

# ── 2. Feature Engineering ────────────────────────────────────────────────────
def add_features(df):
    d = df.copy()

    # Returns
    d["return_1d"]  = d["Close"].pct_change(1)
    d["return_3d"]  = d["Close"].pct_change(3)
    d["return_5d"]  = d["Close"].pct_change(5)
    d["return_10d"] = d["Close"].pct_change(10)

    # Moving averages & ratio
    d["sma_5"]  = d["Close"].rolling(5).mean()
    d["sma_20"] = d["Close"].rolling(20).mean()
    d["sma_50"] = d["Close"].rolling(50).mean()
    d["price_to_sma20"] = d["Close"] / d["sma_20"]

    # Volatility
    d["volatility_10d"] = d["return_1d"].rolling(10).std()
    d["volatility_20d"] = d["return_1d"].rolling(20).std()

    # RSI (14-day)
    delta = d["Close"].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    rs    = gain / loss.replace(0, np.nan)
    d["rsi_14"] = 100 - (100 / (1 + rs))

    # MACD
    ema12 = d["Close"].ewm(span=12, adjust=False).mean()
    ema26 = d["Close"].ewm(span=26, adjust=False).mean()
    d["macd"]        = ema12 - ema26
    d["macd_signal"] = d["macd"].ewm(span=9, adjust=False).mean()
    d["macd_hist"]   = d["macd"] - d["macd_signal"]

    # Bollinger Band position
    bb_mid = d["Close"].rolling(20).mean()
    bb_std = d["Close"].rolling(20).std()
    d["bb_position"] = (d["Close"] - bb_mid) / (2 * bb_std)

    # Volume ratio
    d["volume_ratio"] = d["Volume"] / d["Volume"].rolling(20).mean()

    # High-Low range
    d["hl_range"] = (d["High"] - d["Low"]) / d["Close"]

    return d

df = add_features(df)

# ── 3. Target Variable ────────────────────────────────────────────────────────
df["target"] = (df["Close"].shift(-1) > df["Close"]).astype(int)

FEATURES = [
    "return_1d", "return_3d", "return_5d", "return_10d",
    "price_to_sma20", "volatility_10d", "volatility_20d",
    "rsi_14", "macd_hist", "bb_position", "volume_ratio", "hl_range"
]

df = df.dropna(subset=FEATURES + ["target"])

# ── 4. Train / Val / Test Split (chronological) ───────────────────────────────
n = len(df)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

train = df.iloc[:train_end]
val   = df.iloc[train_end:val_end]
test  = df.iloc[val_end:]

X_train, y_train = train[FEATURES], train["target"]
X_val,   y_val   = val[FEATURES],   val["target"]
X_test,  y_test  = test[FEATURES],  test["target"]

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

print(f"\n  Train: {len(train)} days | Val: {len(val)} days | Test: {len(test)} days")

# ── 5. Random Forest Baseline ─────────────────────────────────────────────────
print("\nTraining Random Forest baseline...")
rf = RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=20,
                            random_state=42, n_jobs=-1)
rf.fit(X_train_s, y_train)

# Evaluate on val and test
for split_name, X_s, y_true in [("Validation", X_val_s, y_val), ("Test", X_test_s, y_test)]:
    y_pred = rf.predict(X_s)
    y_prob = rf.predict_proba(X_s)[:, 1]
    acc  = accuracy_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred)
    auc  = roc_auc_score(y_true, y_prob)
    print(f"\n  [{split_name}]  Accuracy: {acc:.4f}  |  F1: {f1:.4f}  |  AUC-ROC: {auc:.4f}")
    if split_name == "Test":
        print("\n  Classification Report (Test Set):")
        print(classification_report(y_true, y_pred, target_names=["Down (0)", "Up (1)"]))

y_test_pred = rf.predict(X_test_s)
y_test_prob = rf.predict_proba(X_test_s)[:, 1]

# ── 6. Simulated Trading Return ───────────────────────────────────────────────
test_prices = test["Close"].values
daily_returns = np.diff(test_prices) / test_prices[:-1]
signals = y_test_pred[:-1]  # 1 = long, 0 = flat
strategy_returns = np.where(signals == 1, daily_returns, 0)
bh_cumret  = np.cumprod(1 + daily_returns) - 1
strat_cumret = np.cumprod(1 + strategy_returns) - 1
print(f"\n  Buy & Hold return (test period):  {bh_cumret[-1]*100:.1f}%")
print(f"  Strategy return  (test period):   {strat_cumret[-1]*100:.1f}%")

# ── 7. Feature Importance ─────────────────────────────────────────────────────
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)

# ── 8. Plots ──────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 12))
fig.suptitle("S&P 500 Trend Prediction — Random Forest Baseline", fontsize=15, fontweight="bold", y=0.98)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.35)

# (A) Cumulative returns
ax1 = fig.add_subplot(gs[0, :2])
ax1.plot(bh_cumret * 100, label="Buy & Hold", color="#2196F3", linewidth=1.5)
ax1.plot(strat_cumret * 100, label="RF Strategy (long on up-signal)", color="#4CAF50", linewidth=1.5)
ax1.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax1.set_title("Cumulative Returns — Test Set", fontweight="bold")
ax1.set_xlabel("Trading Days")
ax1.set_ylabel("Cumulative Return (%)")
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# (B) Feature importances
ax2 = fig.add_subplot(gs[0, 2])
colors = ["#1565C0" if i == 0 else "#42A5F5" for i in range(len(importances))]
importances.plot(kind="barh", ax=ax2, color=colors[::-1])
ax2.invert_yaxis()
ax2.set_title("Feature Importances", fontweight="bold")
ax2.set_xlabel("Importance")
ax2.grid(True, alpha=0.3, axis="x")
ax2.tick_params(axis="y", labelsize=8)

# (C) Confusion matrix
ax3 = fig.add_subplot(gs[1, 0])
cm = confusion_matrix(y_test, y_test_pred)
im = ax3.imshow(cm, cmap="Blues")
ax3.set_xticks([0, 1]); ax3.set_yticks([0, 1])
ax3.set_xticklabels(["Pred Down", "Pred Up"])
ax3.set_yticklabels(["Actual Down", "Actual Up"])
ax3.set_title("Confusion Matrix (Test)", fontweight="bold")
for i in range(2):
    for j in range(2):
        ax3.text(j, i, str(cm[i, j]), ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=14, fontweight="bold")

# (D) ROC Curve
ax4 = fig.add_subplot(gs[1, 1])
RocCurveDisplay.from_predictions(y_test, y_test_prob, ax=ax4, color="#1565C0")
ax4.plot([0, 1], [0, 1], "k--", linewidth=0.8)
ax4.set_title("ROC Curve (Test)", fontweight="bold")
ax4.grid(True, alpha=0.3)

# (E) Prediction probability distribution
ax5 = fig.add_subplot(gs[1, 2])
ax5.hist(y_test_prob[y_test == 0], bins=30, alpha=0.6, color="#EF5350", label="Actual Down", density=True)
ax5.hist(y_test_prob[y_test == 1], bins=30, alpha=0.6, color="#66BB6A", label="Actual Up", density=True)
ax5.axvline(0.5, color="black", linestyle="--", linewidth=1, label="Threshold = 0.5")
ax5.set_title("Predicted Probability Distribution", fontweight="bold")
ax5.set_xlabel("P(Up)")
ax5.set_ylabel("Density")
ax5.legend(fontsize=8)
ax5.grid(True, alpha=0.3)

plt.savefig("/home/jovyan/ECE542 Project T43/outputs/baseline_results.png", dpi=150, bbox_inches="tight")
print("\nPlot saved.")